In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
# TO DO
import os
import glob
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as T

IMG_SIZE = (256, 256)

class SUIMDataset(Dataset):
    def __init__(self, images_dir, masks_dir, transform=None):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.transform = transform

        self.image_paths = sorted(glob.glob(os.path.join(images_dir, "*")))
        self.mask_paths  = sorted(glob.glob(os.path.join(masks_dir, "*")))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        mask = Image.open(self.mask_paths[idx]).convert("L")

        img = img.resize(IMG_SIZE, Image.BILINEAR)
        mask = mask.resize(IMG_SIZE, Image.NEAREST)

        img = T.ToTensor()(img)
        mask = torch.tensor(np.array(mask), dtype=torch.long)

        mask = remap_mask(mask)

        if self.transform:
            img = self.transform(img)

        return img, mask


data_path = os.path.join(path, "dataset")
images_dir = os.path.join(data_path, "images")
masks_dir = os.path.join(data_path, "masks")

transform = T.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])

dataset = SUIMDataset(images_dir, masks_dir, transform=transform)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)

print("Total:", len(dataset))
print("Train:", len(train_dataset))
print("Val:", len(val_dataset))


# Display some samples
imgs, masks = next(iter(train_loader))

plt.figure(figsize=(12, 6))
for i in range(3):
    plt.subplot(2, 3, i+1)
    plt.imshow(imgs[i].permute(1,2,0).numpy())
    plt.axis("off")
    plt.title("Image")

    plt.subplot(2, 3, i+4)
    plt.imshow(masks[i].numpy(), cmap="gray")
    plt.axis("off")
    plt.title("Mask")

plt.show()


In [ ]:
# TO DO
!pip -q install segmentation-models-pytorch

import torch
import segmentation_models_pytorch as smp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = smp.Unet(
    encoder_name="efficientnet-b1",
    encoder_weights="imagenet",
    in_channels=3,
    classes=8,
    activation=None
).to(device)

print(device)


In [ ]:
# TO DO
import torch
from tqdm import tqdm

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0

    for imgs, masks in tqdm(loader, leave=False):
        imgs = imgs.to(device)
        masks = masks.to(device)  # (B,H,W) long

        logits = model(imgs)      # (B,C,H,W)
        loss = criterion(logits, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0

    for imgs, masks in loader:
        imgs = imgs.to(device)
        masks = masks.to(device)

        logits = model(imgs)
        loss = criterion(logits, masks)

        total_loss += loss.item()

    return total_loss / len(loader)


In [ ]:
# TO DO
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

num_epochs = 10
train_losses = []
val_losses = []

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate_one_epoch(model, val_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch [{epoch+1}/{num_epochs}]  Train Loss: {train_loss:.4f}  Val Loss: {val_loss:.4f}")

plt.figure(figsize=(8,5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()
plt.grid()
plt.show()


In [ ]:
# TO DO
import torch
import matplotlib.pyplot as plt
import numpy as np
import random

def denormalize(img):
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img = img.permute(1,2,0).cpu().numpy()
    img = img * std + mean
    return np.clip(img, 0, 1)

model.eval()

num_samples = 5
indices = random.sample(range(len(val_dataset)), num_samples)

for idx in indices:
    img, mask = val_dataset[idx]

    with torch.no_grad():
        pred = model(img.unsqueeze(0).to(device))
        pred_mask = torch.argmax(pred, dim=1).squeeze(0).cpu()

    plt.figure(figsize=(15,5))

    plt.subplot(1,3,1)
    plt.imshow(denormalize(img))
    plt.title("Image")
    plt.axis("off")

    plt.subplot(1,3,2)
    plt.imshow(mask.cpu(), cmap="gray")
    plt.title("Ground Truth")
    plt.axis("off")

    plt.subplot(1,3,3)
    plt.imshow(pred_mask, cmap="gray")
    plt.title("Prediction")
    plt.axis("off")

    plt.show()
